In [ ]:
%%HTML
<style>

.CodeMirror {
    width: 85%;
    font-size:1.4em;
}

.container {
    width: 80% !important;
}

.rendered_html {
  font-size:1.4em;
}
.rendered_html table, .rendered_html th, .rendered_html tr, .rendered_html td {
     font-size: 100%;
}


body {
  font-family: Ubuntu;
  background: #F0F0F0;
  background-color: #F0F0F0;
}


.reveal h1,
.reveal h2,
.reveal h3,
.reveal h4,
.reveal h5,
.reveal h6 {
  margin: 0 0 20px 0;
  color: #2a2eec;
  font-family: Ubuntu;
  line-height: 0.9em;
  letter-spacing: 0.02em;
  text-transform: none;
  text-shadow: none;
}

.reveal blockquote {
  display: block;
  position: relative;
  background: #fa7c17;
  border-radius: 15px;
  box-shadow: 0px 0px 2px rgba(0, 0, 0, 0.2);
  font-weight: bold;
}

</style>

In [ ]:
#ibrerías

from PIL import Image
import numpy as np
import os

# Midiendo la divergencia del haz de un láser.

La idea principal es medir el efecto de la divergencia de un diodo láser, como se muestra en la hora de datos de un Mitsubishi:
<img src="figjn/divergence.jpg" width="700" align = "center">

Las dos últimas líneas nos informan ángulos de divergencia.

El láser que nos convoca tiene varias lentes luego de la salida, por lo que no podemos buscar en la hoja de datos del diodo, sino que tendremos que levantar la curva. La idea es la siguiente:

* Sacar fotografías del láser en una pantalla perpendicular, para diferentes distancias de disparo.

* Cada foto se renombra con la distancia de disparo.

* El algoritmo busca un percentil en los píxels para dividir lo que **es** de lo que **no es** parte de la imagen del láser.

* Dividiendo píxels, se pueden medir los diámetros horizontal y vertical en píxels, que luego serán pasados a unidades de distancia utilizando una calibración.



In [ ]:
#ingresar directorio de imágenes
dir_imagenes = '/home/juan/Documentos/Docencia/FEXPgit/FexpI/Code/PhotoContrastPy/DataSample/'
os.chdir(dir_imagenes);
os.getcwd() #comprueba que estemos en el directorio de las fotos.

In [ ]:
def medir_laser(nombre_archivo, percentil=99.9, margen=20):
    # Cargar imagen
    img = Image.open(nombre_archivo)
    imgnum = np.asarray(img)
    verde = imgnum[:,:,1]
   
    # Umbral automático con percentil
    umbral = np.percentile(verde, percentil)
    mascara = verde > umbral
   
    # Encontrar posiciones del láser
    posiciones = np.where(mascara)
    filas = posiciones[0]
    columnas = posiciones[1]
   
    # Calcular tamaño
    alto_laser = filas.max() - filas.min() + 1
    ancho_laser = columnas.max() - columnas.min() + 1
   
    #print(f"Archivo: {nombre_archivo}")
    print(f"{umbral}", end = ",")
    print(f"{mascara.sum()}", end = ",")
    print(f"{alto_laser}", end = ",")
    print(f"{ancho_laser}")
    
   


In [ ]:
#Percentil usado para la detección. Extensión de imágenes.
percentil = 99.9
extension_imagenes = '.jpeg'

In [ ]:


print("#carpeta:" + os.getcwd());
print("#percentil utilizado:" + f"{percentil}")
print("#Extensión de imágenes:" + f"{extension_imagenes}")

print("d,umbral,PixelesDetectados,vertD,horD")

for x in os.listdir():
    if x.endswith(extension_imagenes):
        # Prints only text file present in My Folder
        print(x.removesuffix(extension_imagenes) + ',', end = "")
        medir_laser(x,percentil)
        

# Ajuste con gaussianas

In [ ]:
using Pkg
Pkg.add("Optim")

In [1]:
using Distributions, Optim

# generate some sample data    
true_d = MvNormal([1.0, 0.0], [2.0  1.0; 1.0 3.0])
const xr = -3:0.1:3
const yr = -3:0.1:3
const s = 100.0
const m = [s * pdf(true_d, [x, y]) for x in xr, y in yr];

In [ ]:
#using Plots
#heatmap(m)


In [2]:
decode(x) = (mu=x[1:2], sig=[x[3] x[4]; x[4] x[5]], s=x[6])

function objective(x)
    mu, sig, s = decode(x)
    try # sig might be infeasible so we have to handle this case
        est_d = MvNormal(mu, sig)
        ref_m = [s * pdf(est_d, [x, y]) for x in xr, y in yr]
        sum((a-b)^2 for (a,b) in zip(ref_m, m))
    catch
        sum(m)
    end
end

objective (generic function with 1 method)

In [3]:
# test for an example starting pplot(m,oint
result = optimize(objective, [1.0, 0.0, 1.0, 0.0, 1.0, 1.0])
decode(result.minimizer)

(mu = [0.7369184647534524, -0.01373046259429404], sig = [-0.05612460961099186 0.018355027093635082; 0.018355027093635082 0.3355621786400286], s = 2.926757429355132)

In [ ]:
a = MvNormal([1.0, 0.0, 2.00], [2.0 1.0 1.0; 1.0 3.0 1.0; 1.0 1.0 1.0])

# Cargar imágenes.

In [ ]:
using FileIO, Images

In [ ]:
cd("/home/juan/Documentos/Docencia/fexpi/2025/IL/")
pwd()

In [ ]:
# specify the path to your local image file
img_path = "1000.jpg"
img = load(img_path)

In [ ]:
#intentamos contraste?
typeof(img)

In [ ]:
img_CHW = channelview(img); # 3 * 2 * 2

In [ ]:
verde = img_CHW[2,:,:]


In [ ]:
vl = verde .* (verde .> 0.01);
hi = vec(vl);


In [ ]:
hi = deleteat!(hi, hi .== 0.000);

In [ ]:
typeof(hi)
length(hi)

In [ ]:
using Plots

histogram(hi)

In [ ]:
typeof(hi)

In [ ]:
findmax(hi)

In [ ]:
findmax(vl)

In [ ]:
sum(vec(verde))

In [ ]:
sum(hi)

In [ ]:
result = optimize(verde, [1.0, 0.0, 1.0, 0.0, 1.0, 1.0])
decode(result.minimizer)